# Population PSTHs grouped by early mean speed

Align all recorded neurons to **movement onset**, split tracking/playback, and
group bouts by `early_mean_speed` in **position units/s**: [0, 0.5), [0.5, 1),
[1, 1.5), [1.5, 2), [2, 3), [3, 4), [4, 6), [6, 10), and [10, infinity). Exact boundaries go in the higher group.
The speed label uses the saved early interval (1 s or earlier qualifying immobility).

For each neuron, average its baseline-corrected bout traces within each speed
group and condition; then average those neuron PSTHs **with equal neuron weight**.
Shading is SEM across recorded unit PSTHs, not independent animals or bouts.
Group membership can differ across sessions: counts of bouts, units and sessions
are shown below. Opposite tuning across neurons can cancel in the population mean.

Baseline subtraction and 10 ms Gaussian smoothing follow `perimotor_psth.ipynb`.
Set the baseline to `None` for raw rates, or smoothing to zero to disable it.
Symmetric smoothing can spread responses across onset. Conditions are assigned
at onset; transition-crossing windows are not separately excluded. Complete
neural windows are required. Original bout IDs are retained. No data files are
written. Run top to bottom with the project .venv kernel.


The additional 100 ms analysis uses **unsmoothed spike counts**, subtracting the
same per-bout/neuron baseline. Its two heatmaps share one zero-centered activity
color scale over the full displayed interval. No sound-related interval is masked.
The companion selector shows mean activity +/- SEM for one 100 ms interval at a
time. These population summaries do not measure single-bout decoding accuracy.


In [ ]:
from pathlib import Path
import json
import pickle
import sys
from importlib import reload
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SESSION_INDICES = None  # None = all saved sessions; e.g. [0] for one session.
SPEED_EDGES = [0, 0.5, 1, 1.5, 2, 3, 4, 6, 10, np.inf]
SPEED_CMAP = "coolwarm"  # Shared blue (slow) to red (fast) scale in both panels.
WINDOW_S = (-1.2, 1.2)  # Includes baseline and smoothing margins.
DISPLAY_WINDOW_S = (-0.5, 1.0)
BASELINE_WINDOW_S = (-1.0, -0.7)  # None = raw firing rates.
BIN_WIDTH_S = 0.005
SMOOTHING_SIGMA_S = 0.01
SHOW_SEM = True
QUANT_BIN_WIDTH_S = 0.1  # Direct count-based means; no Gaussian smoothing.
ACTIVITY_CMAP = "coolwarm"  # Cell color encodes firing-rate change in heatmaps.

cwd = Path.cwd().resolve()
repo_dir = next((candidate for parent in (cwd, *cwd.parents)
                 for candidate in (parent, parent / "etheremin_project")
                 if (candidate / "movement_bouts/psth.py").is_file()), None)
if repo_dir is None:
    raise FileNotFoundError("Run from the eTheremin workspace or repository.")
sys.path.insert(0, str(repo_dir / "movement_bouts"))
import psth
reload(psth)
from psth import session_neuron_psths
data_dir = repo_dir.parent / "data"
bout_dir = data_dir / "derived/movement_bouts"


In [ ]:
movement_bouts = pd.read_pickle(bout_dir / "movement_bouts.pkl")
session_info = pd.read_pickle(bout_dir / "session_info.pkl")
parameters = json.loads((bout_dir / "parameters.json").read_text(encoding="utf-8"))
DT = parameters["DT"]
indices = session_info.index if SESSION_INDICES is None else pd.Index(SESSION_INDICES)
if not len(indices) or not indices.is_unique or not indices.isin(session_info.index).all():
    raise ValueError("Choose nonempty, unique saved session indices.")
edges = np.asarray(SPEED_EDGES, dtype=float)
if edges.ndim != 1 or len(edges) < 2 or edges[0] != 0 or not np.isposinf(edges[-1]) or not np.all(np.diff(edges) > 0):
    raise ValueError("Speed edges must increase from 0 to positive infinity.")
speed_labels = [f"{lo:g}–{hi:g}" if np.isfinite(hi) else f"≥{lo:g}"
                for lo, hi in zip(edges[:-1], edges[1:])]
bouts = movement_bouts.loc[movement_bouts["idx"].isin(indices)
                           & movement_bouts["condition"].isin([0, 1])].copy()
valid_speed = np.isfinite(bouts["early_mean_speed"]) & bouts["early_mean_speed"].ge(0)
print(f"Excluded {int((~valid_speed).sum())} bouts with missing/nonfinite/negative early speed.")
bouts = bouts.loc[valid_speed].copy()
bouts["speed_group"] = pd.cut(bouts["early_mean_speed"], bins=edges, right=False, labels=False)
print(f"{len(bouts):,} speed-labelled bouts; {len(indices)} session entries. "
      f"Saved speed cap: {parameters['EARLY_SPEED_WINDOW_S']:g} s or earlier immobility.")
display(bouts.groupby(["condition", "speed_group"]).size().unstack("condition", fill_value=0)
        .reindex(range(len(speed_labels)), fill_value=0).rename(index=dict(enumerate(speed_labels)), columns={0: "Tracking", 1: "Playback"}))


In [ ]:
# Read one source collection at a time; retain only per-neuron group averages.
settings = dict(dt=DT, window_s=WINDOW_S, bin_width_s=BIN_WIDTH_S,
                baseline_window_s=BASELINE_WINDOW_S, smoothing_sigma_s=SMOOTHING_SIGMA_S)
centers = WINDOW_S[0] + (np.arange(round((WINDOW_S[1] - WINDOW_S[0]) / BIN_WIDTH_S)) + .5) * BIN_WIDTH_S
traces = {(condition, group): [] for condition in (0, 1) for group in range(len(speed_labels))}
units = {key: [] for key in traces}
session_rows = []
quant_settings = {**settings, "bin_width_s": QUANT_BIN_WIDTH_S, "smoothing_sigma_s": 0}
quant_edges = np.round(WINDOW_S[0] + np.arange(round((WINDOW_S[1] - WINDOW_S[0]) / QUANT_BIN_WIDTH_S) + 1) * QUANT_BIN_WIDTH_S, 10)
quant_edges[np.isclose(quant_edges, 0)] = 0
quant_centers = (quant_edges[:-1] + quant_edges[1:]) / 2
quant_traces = {key: [] for key in traces}
for feature_file, sessions in session_info.loc[indices].groupby("feature_file", sort=False):
    print(f"Loading {feature_file.replace('_feature_ss', '_data_ss')} ...", flush=True)
    with (data_dir / feature_file.replace("_feature_ss", "_data_ss")).open("rb") as stream:
        recordings = pickle.load(stream)
    with (data_dir / feature_file.replace("_feature_ss", "_neuron_idx_ss")).open("rb") as stream:
        ids_collection = pickle.load(stream)
    for idx, info in sessions.iterrows():
        neural = recordings[int(info["session"])]
        neuron_ids = ids_collection[int(info["session"])]
        if neural.shape != (len(neuron_ids), int(info["n_bins"])) or len(set(neuron_ids)) != len(neuron_ids):
            raise ValueError(f"Recording dimensions or neuron IDs are invalid for idx={idx}.")
        for group in range(len(speed_labels)):
            selected = bouts.loc[bouts["idx"].eq(idx) & bouts["speed_group"].eq(group)]
            if len(selected):
                averages, times, counts = session_neuron_psths(neural, selected, **settings)
                np.testing.assert_allclose(times, centers)
                quant_averages, quant_times, quant_counts = session_neuron_psths(neural, selected, **quant_settings)
                np.testing.assert_allclose(quant_times, quant_centers)
                if quant_counts != counts:
                    raise ValueError("Smoothed and binned views must retain the same bouts.")
            else:
                averages = {condition: np.empty((0, len(centers))) for condition in (0, 1)}
                counts = {0: 0, 1: 0}
            for condition in (0, 1):
                key = (condition, group)
                if len(averages[condition]):
                    traces[key].append(averages[condition])
                    quant_traces[key].append(quant_averages[condition])
                    units[key].extend((idx, neuron_id) for neuron_id in neuron_ids)
                session_rows.append(dict(idx=idx, condition=condition, speed_group=group,
                                         n_units=len(averages[condition]), n_bouts=counts[condition],
                                         n_excluded=int(selected["condition"].eq(condition).sum()) - counts[condition]))
    del neural, recordings, ids_collection

speed_population_psths = {
    key: pd.DataFrame(np.concatenate(values) if values else np.empty((0, len(centers))),
                      index=pd.MultiIndex.from_tuples(units[key], names=["idx", "neuron_id"]),
                      columns=pd.Index(centers, name="time_s"))
    for key, values in traces.items()
}
session_counts = pd.DataFrame(session_rows)
population_counts = session_counts.groupby(["condition", "speed_group"])[["n_bouts", "n_units", "n_excluded"]].sum()
population_counts["n_sessions"] = session_counts.loc[session_counts["n_bouts"].gt(0)].groupby(["condition", "speed_group"])["idx"].nunique().reindex(population_counts.index, fill_value=0)
display(population_counts.rename(index={0: "Tracking", 1: "Playback"}, level="condition")
        .rename(index=dict(enumerate(speed_labels)), level="speed_group"))

# These tables retain one row per recorded neuron, allowing honest SEM across units.
binned_neuron_psths = {
    key: pd.DataFrame(np.concatenate(values) if values else np.empty((0, len(quant_centers))),
                      index=speed_population_psths[key].index.copy(),
                      columns=pd.Index(quant_centers, name="time_s"))
    for key, values in quant_traces.items()
}


## Population response by early mean speed

Each curve averages neurons equally within the indicated group and condition.
Both panels use the same coolwarm scale: blue for low speed, neutral in the middle,
and red for high speed. Counts are based on
complete neural windows. Units are recorded session units; the SEM does not
account for correlations within sessions or animals. Replotting needs no disk reads.


In [ ]:
visible = (centers >= DISPLAY_WINDOW_S[0]) & (centers < DISPLAY_WINDOW_S[1])
if not visible.any():
    raise ValueError("Display window contains no PSTH bins.")
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), sharex=True, sharey=True, layout="constrained")
colors = plt.get_cmap(SPEED_CMAP)(np.linspace(0, 1, len(speed_labels)))
for ax, condition, name in zip(axes, (0, 1), ("Tracking", "Playback")):
    for group, (label, color) in enumerate(zip(speed_labels, colors)):
        values = speed_population_psths[(condition, group)].to_numpy()[:, visible]
        count = population_counts.loc[(condition, group)]
        legend = f"{label}: {int(count['n_bouts']):,} bouts, {len(values):,} units"
        if not len(values):
            ax.plot([], [], color=color, label=legend)
            continue
        mean = values.mean(axis=0)
        ax.plot(centers[visible], mean, color=color, linewidth=1.8, label=legend)
        if SHOW_SEM and len(values) > 1:
            sem = values.std(axis=0, ddof=1) / np.sqrt(len(values))
            ax.fill_between(centers[visible], mean - sem, mean + sem, color=color, alpha=.13)
    ax.axvline(0, color=".4", linestyle="--", linewidth=1)
    if settings["baseline_window_s"] is not None:
        ax.axhline(0, color=".65", linewidth=.7)
    ax.set(title=name, xlabel="Time from movement onset (s)", xlim=DISPLAY_WINDOW_S)
    ax.legend(title="Early mean speed (position units/s)", frameon=False, fontsize=8, loc="upper left")
    ax.grid(axis="y", alpha=.12)
axes[0].set_ylabel("Change in firing rate (spikes/s)" if settings["baseline_window_s"] is not None else "Firing rate (spikes/s)")
baseline = settings["baseline_window_s"]
baseline_text = f"baseline [{baseline[0]:g}, {baseline[1]:g}) s" if baseline is not None else "raw rates"
fig.suptitle(f"Population PSTH grouped by early mean speed | {len(indices)} session entries\n"
             f"Equal neuron weight | {baseline_text} | Gaussian sigma {settings['smoothing_sigma_s'] * 1000:g} ms", fontsize=12)
plt.show()


## Mean activity in successive 100 ms intervals

Rows are the finer early-speed groups; columns are nonoverlapping time intervals.
Colors encode mean activity in spikes/s, with **one shared scale** for both panels.
The dashed line marks movement onset. Gray rows have no contributing neurons.
Row labels give complete bout and unit counts; session coverage is in the table
above. Quantification uses unsmoothed counts, with the same baseline subtraction
and equal-neuron weighting as the PSTHs. `binned_summary` retains exact means,
SEMs, interval bounds and coverage. All displayed bins are kept, including the
large post-onset response; no pre-onset zoom or masking is applied.


In [ ]:
from matplotlib.colors import TwoSlopeNorm

quant_visible = (quant_edges[:-1] >= DISPLAY_WINDOW_S[0] - 1e-9) & (quant_edges[1:] <= DISPLAY_WINDOW_S[1] + 1e-9)
quant_indices = np.flatnonzero(quant_visible)
if not len(quant_indices):
    raise ValueError("No complete quantification bins fit inside the display window.")
bin_starts, bin_stops = quant_edges[:-1][quant_visible], quant_edges[1:][quant_visible]
bin_means, bin_sems = {}, {}
summary_rows = []
for condition in (0, 1):
    bin_means[condition] = np.full((len(speed_labels), len(quant_indices)), np.nan)
    bin_sems[condition] = np.full_like(bin_means[condition], np.nan)
    for group in range(len(speed_labels)):
        values = binned_neuron_psths[(condition, group)].to_numpy()[:, quant_visible]
        counts = population_counts.loc[(condition, group)]
        if len(values):
            bin_means[condition][group] = values.mean(axis=0)
        if len(values) > 1:
            bin_sems[condition][group] = values.std(axis=0, ddof=1) / np.sqrt(len(values))
        for column, (start, stop) in enumerate(zip(bin_starts, bin_stops)):
            summary_rows.append(dict(condition=condition, speed_group=group,
                                     speed_label=speed_labels[group], bin_start_s=start, bin_stop_s=stop,
                                     mean_rate=bin_means[condition][group, column],
                                     sem_rate=bin_sems[condition][group, column],
                                     n_units=len(values), n_bouts=int(counts["n_bouts"]),
                                     n_sessions=int(counts["n_sessions"])))
binned_summary = pd.DataFrame(summary_rows)
finite = np.concatenate([matrix[np.isfinite(matrix)] for matrix in bin_means.values()])
activity_limit = max(float(np.abs(finite).max()), .01) if len(finite) else 1.
activity_norm = TwoSlopeNorm(vmin=-activity_limit, vcenter=0, vmax=activity_limit)
activity_cmap = plt.get_cmap(ACTIVITY_CMAP).with_extremes(bad=".85")
fig_heatmap, axes_heatmap = plt.subplots(1, 2, figsize=(16, 7), sharex=True, layout="constrained")
for ax, condition, name in zip(axes_heatmap, (0, 1), ("Tracking", "Playback")):
    image = ax.imshow(np.ma.masked_invalid(bin_means[condition]), origin="lower", aspect="auto",
                      interpolation="nearest", cmap=activity_cmap, norm=activity_norm,
                      extent=(bin_starts[0], bin_stops[-1], -.5, len(speed_labels) - .5))
    row_labels = [f"{label}\n{int(population_counts.loc[(condition, group), 'n_bouts']):,} bouts; "
                  f"{int(population_counts.loc[(condition, group), 'n_units']):,} units"
                  for group, label in enumerate(speed_labels)]
    ax.set_yticks(range(len(speed_labels)), row_labels, fontsize=8)
    ax.set_xticks(bin_starts[::2], [f"{value:g}" for value in bin_starts[::2]])
    ax.axvline(0, color="black", linestyle="--", linewidth=1.1)
    ax.set(title=name, xlabel="Time from movement onset (s)", ylabel="Early mean speed (position units/s)")
activity_label = "Mean change in firing rate (spikes/s)" if quant_settings["baseline_window_s"] is not None else "Mean firing rate (spikes/s)"
fig_heatmap.colorbar(image, ax=axes_heatmap, label=activity_label, shrink=.85)
quant_baseline = quant_settings["baseline_window_s"]
quant_baseline_text = f"baseline [{quant_baseline[0]:g}, {quant_baseline[1]:g}) s" if quant_baseline is not None else "raw rates"
fig_heatmap.suptitle(f"Population activity in {quant_settings['bin_width_s'] * 1000:g} ms intervals | equal neuron weight\n"
                     f"Unsmoothed counts | {quant_baseline_text} | shared color scale", fontsize=12)
plt.show()


## Speed dependence in one selected interval

The static preview starts with the first interval at/after movement onset.
Points show neuron means +/- SEM, with the same blue-to-red speed colors in
both panels. The y-axis is shared between conditions for the selected interval;
speed groups are categorical, so their unequal widths are explicit in the labels.
The table gives numerical values and sample counts. The optional selector below
redraws from the computed summaries, without reading recordings again.


In [ ]:
def plot_speed_interval(interval_index):
    if not 0 <= interval_index < len(bin_starts):
        raise ValueError("Choose a displayed interval index.")
    fig_interval, axes_interval = plt.subplots(1, 2, figsize=(15, 4.8), sharey=True, layout="constrained")
    x = np.arange(len(speed_labels))
    group_colors = plt.get_cmap(SPEED_CMAP)(np.linspace(0, 1, len(speed_labels)))
    for ax, condition, name in zip(axes_interval, (0, 1), ("Tracking", "Playback")):
        mean, sem = bin_means[condition][:, interval_index], bin_sems[condition][:, interval_index]
        ax.plot(x, mean, color=".65", linewidth=1)
        ax.errorbar(x, mean, yerr=sem, fmt="none", ecolor=".4", capsize=3, linewidth=1)
        ax.scatter(x, mean, c=group_colors, s=45, zorder=3)
        ax.axhline(0, color=".7", linewidth=.8)
        ax.set_xticks(x, speed_labels, rotation=35, ha="right")
        ax.set(title=name, xlabel="Early mean speed group (position units/s)")
        ax.grid(axis="y", alpha=.15)
    axes_interval[0].set_ylabel(activity_label)
    fig_interval.suptitle(f"Mean activity [{bin_starts[interval_index]:g}, {bin_stops[interval_index]:g}) s after movement onset\n"
                          "Unsmoothed counts | mean +/- SEM across recorded units", fontsize=12)
    return fig_interval, axes_interval

DEFAULT_INTERVAL = int(np.searchsorted(bin_starts, 0))
DEFAULT_INTERVAL = min(DEFAULT_INTERVAL, len(bin_starts) - 1)
fig_interval, axes_interval = plot_speed_interval(DEFAULT_INTERVAL)
plt.show()
display(binned_summary.loc[np.isclose(binned_summary["bin_start_s"], bin_starts[DEFAULT_INTERVAL])]
        .replace({"condition": {0: "Tracking", 1: "Playback"}}).round(3))


In [ ]:
# Run this cell to activate the in-memory interval selector.
import ipywidgets as widgets
from IPython.display import clear_output

interval_slider = widgets.SelectionSlider(
    options=[(f"[{start:g}, {stop:g}) s", index) for index, (start, stop) in enumerate(zip(bin_starts, bin_stops))],
    value=DEFAULT_INTERVAL, description="Time bin:", continuous_update=False,
    layout=widgets.Layout(width="95%"),
)
interval_output = widgets.Output()

def redraw_interval(change):
    with interval_output:
        clear_output(wait=True)
        figure, _ = plot_speed_interval(interval_slider.value)
        display(figure)
        plt.close(figure)
        display(binned_summary.loc[np.isclose(binned_summary["bin_start_s"], bin_starts[interval_slider.value])]
                .replace({"condition": {0: "Tracking", 1: "Playback"}}).round(3))

interval_slider.observe(redraw_interval, names="value")
display(widgets.VBox([interval_slider, interval_output]))
print("Move the selector to redraw the interval plot and table below. No new disk reads are needed.")
